# 📘 Chapter 1: Common Conventions and API Elements
**Referensi Buku:** *scikit-learn Cookbook, Third Edition*

---
## 1. Pendahuluan
Kekuatan utama `scikit-learn` terletak pada desain API-nya yang sangat konsisten. Setelah Anda memahami cara kerja satu algoritma (misalnya dengan memanggil `.fit()` dan `.predict()`), Anda secara otomatis tahu cara menggunakan ratusan algoritma lainnya di library ini.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Membuat dataset dummy sederhana untuk klasifikasi
X, y = make_classification(n_samples=100, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Estimator & Predictor API
Setiap algoritma *machine learning* di scikit-learn disebut sebagai **Estimator**. Estimator belajar dari data melalui metode `fit()`. Setelah belajar, ia menjadi **Predictor** yang bisa menebak data baru menggunakan `predict()`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Inisialisasi Estimator
clf = RandomForestClassifier(random_state=42)

# Estimator API: Belajar dari data
clf.fit(X_train, y_train)
print("Model telah berhasil dilatih (fitted).\n")

# Predictor API: Membuat prediksi pada data pengujian
predictions = clf.predict(X_test)
print("Prediksi kelas untuk 5 data pertama:", predictions[:5])

# Beberapa prediktor juga bisa mengeluarkan probabilitas
probabilities = clf.predict_proba(X_test)
print("Probabilitas [Kelas 0, Kelas 1] untuk data pertama:", probabilities[0])

## 3. Transformer API
**Transformer** adalah objek yang memodifikasi data (seperti mengubah skala atau mengisi nilai kosong). Transformer menggunakan metode `transform()`. Metode gabungan `fit_transform()` sering digunakan pada data latih agar lebih efisien.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit dan Transform digabung untuk data training
X_train_scaled = scaler.fit_transform(X_train)

# Hanya Transform untuk data test (agar tidak terjadi data leakage)
X_test_scaled = scaler.transform(X_test)

print("Rata-rata fitur pertama sebelum scaling:", X_train[:, 0].mean())
print("Rata-rata fitur pertama sesudah scaling:", X_train_scaled[:, 0].mean(), "(Mendekati Nol)")

## 4. Pipelines
Pipeline adalah fitur revolusioner dari scikit-learn. Ia memungkinkan kita menggabungkan beberapa langkah Transformer dan satu Estimator di akhir menjadi sebuah kesatuan. Ini membuat kode jauh lebih rapi dan mencegah kebocoran data (*data leakage*).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Membuat Pipeline: Scaling -> Klasifikasi SVM
workflow = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(probability=True))
])

# Kita cukup memanggil fit pada pipeline!
workflow.fit(X_train, y_train)

print("Akurasi Pipeline pada Test Set:", workflow.score(X_test, y_test))

## 5. Model Persistence (Menyimpan Model)
Melatih model bisa memakan waktu berjam-jam. Kita bisa menyimpan model yang sudah dilatih (termasuk pipeline-nya) ke dalam file menggunakan modul eksternal bernama `joblib`.

In [ ]:
import joblib
import os

# Menyimpan model ke disk
model_filename = 'my_pipeline_model.pkl'
joblib.dump(workflow, model_filename)
print(f"Model berhasil disimpan ke: {model_filename}")

# Memuat kembali model dari disk
loaded_model = joblib.load(model_filename)
print("Akurasi Model yang dimuat ulang:", loaded_model.score(X_test, y_test))

# Bersihkan file (opsional)
if os.path.exists(model_filename):
    os.remove(model_filename)